# Bài 14: Các bộ lọc phi tuyến (Nonlinear Filters)

Mọi bộ lọc từ đầu đến giờ (hộp, Gauss, Sobel, Laplace, ...) đều là tích chập: một tổ hợp tuyến tính cố định của các giá trị điểm ảnh lân cận. Nhưng bộ lọc tuyến tính bị giới hạn về những gì chúng làm được &mdash; chúng không thể suy xét xem điểm ảnh lân cận nào là "ngoại lai (outlier)" hay điểm nào "nằm ở phía bên kia của một biên". Bài này xem xét các bộ lọc phá vỡ tính tuyến tính đó: **lọc trung vị (median filtering)**, **co/giãn** ảnh xám **dưới dạng bộ lọc min/max**, và **bộ lọc song phương (bilateral filter)** bảo toàn biên.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Lọc trung vị: chống chịu tốt với ngoại lai

Bộ lọc trung vị thay mỗi điểm ảnh bằng **trung vị (median)** (không phải trung bình) của vùng lân cận. Trong khi một giá trị điểm ảnh sai lệch nghiêm trọng duy nhất (ngoại lai) có thể ảnh hưởng mạnh đến giá trị trung bình, nó hầu như không làm trung vị dịch chuyển, nếu có &mdash; nên lọc trung vị đặc biệt hiệu quả với **nhiễu muối tiêu (salt-and-pepper noise)** (các điểm ảnh cô lập bị đẩy ngẫu nhiên thành đen hoặc trắng), loại nhiễu mà làm trơn tuyến tính xử lý kém.

In [ ]:
clean = np.zeros((120, 120), dtype=np.uint8)
cv2.rectangle(clean, (25, 25), (95, 95), 200, -1)

rng = np.random.default_rng(2)
salt_pepper = clean.copy()
hit = rng.random(clean.shape) < 0.06
salt_pepper[hit & (rng.random(clean.shape) < 0.5)] = 0
salt_pepper[hit & (rng.random(clean.shape) >= 0.5)] = 255

median_result = cv2.medianBlur(salt_pepper, 5)
gaussian_result = cv2.GaussianBlur(salt_pepper, (5, 5), sigmaX=1.5)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [salt_pepper, gaussian_result, median_result],
                          ['Salt-and-pepper noise', 'Gaussian blur', 'Median filter']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('mean abs error vs. clean image:')
print(f'  Gaussian blur : {np.abs(gaussian_result.astype(int) - clean.astype(int)).mean():.2f}')
print(f'  Median filter : {np.abs(median_result.astype(int) - clean.astype(int)).mean():.2f}')

Làm mờ Gauss làm nhòe giá trị cực đoan của mỗi điểm ảnh bị hỏng sang các điểm lân cận, để lại các đốm lấm tấm mờ khắp nơi. Bộ lọc trung vị đơn giản là để nhiều điểm lân cận tốt "bỏ phiếu áp đảo" một điểm ảnh xấu cô lập, loại bỏ gần như toàn bộ nhiễu trong khi giữ các cạnh sắc nét.

## Bộ lọc min và max: co và giãn ảnh xám

Bài 3 đã giới thiệu phép co và giãn trên ảnh *nhị phân*. Trên ảnh **xám**, chúng tổng quát hóa một cách tự nhiên: phép co thay mỗi điểm ảnh bằng giá trị **nhỏ nhất** trong vùng lân cận, và phép giãn bằng giá trị **lớn nhất**. Cả hai đều phi tuyến (không phải tổng có trọng số), và cả hai đều bảo toàn biên theo một cách khác với làm mờ &mdash; chúng dịch chuyển biên thay vì làm mờ biên.

In [ ]:
def min_filter(image, ksize):
    r = ksize // 2
    padded = cv2.copyMakeBorder(image, r, r, r, r, cv2.BORDER_REPLICATE)
    out = np.zeros_like(image)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            out[i, j] = padded[i:i + ksize, j:j + ksize].min()
    return out

small = clean[::4, ::4]  # smaller image so the manual pixel loop stays fast
mine = min_filter(small, 3)
reference = cv2.erode(small, np.ones((3, 3), np.uint8))
print('manual min filter == cv2.erode?', np.array_equal(mine, reference))

## Vấn đề mà làm mờ tuyến tính không giải quyết được: làm trơn *mà không* vượt qua biên

Làm mờ Gauss đối xử với mọi điểm lân cận như nhau bất kể giá trị của nó khác biệt đến đâu &mdash; đó chính là điều khiến nó làm mờ xuyên qua biên. **Bộ lọc song phương** khắc phục điều này bằng cách đánh trọng số mỗi điểm lân cận theo *hai* yếu tố: nó gần đến đâu về mặt không gian (giống làm mờ Gauss) **và** cường độ của nó giống điểm ảnh trung tâm đến đâu:

$$I'(p) = \frac{1}{W_p}\sum_{q \in N(p)} G_{\sigma_s}(\|p - q\|) \cdot G_{\sigma_r}(|I(p) - I(q)|) \cdot I(q)$$

trong đó $W_p$ chuẩn hóa các trọng số để tổng bằng 1. Các điểm lân cận vừa gần về không gian *vừa* có cường độ tương tự sẽ có trọng số cao (được làm trơn cùng nhau); các điểm lân cận gần về không gian nhưng có cường độ rất khác (tức là ở phía bên kia của một biên) bị giảm trọng số gần như về 0, nên biên được giữ lại.

In [ ]:
def bilateral_filter(image, radius, sigma_spatial, sigma_range):
    image_f = image.astype(np.float64)
    ys, xs = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    spatial_weight = np.exp(-(xs**2 + ys**2) / (2 * sigma_spatial**2))
    padded = cv2.copyMakeBorder(image_f, radius, radius, radius, radius, cv2.BORDER_REFLECT101)

    out = np.zeros_like(image_f)
    h, w = image.shape
    for i in range(h):
        for j in range(w):
            patch = padded[i:i + 2 * radius + 1, j:j + 2 * radius + 1]
            range_weight = np.exp(-(patch - image_f[i, j])**2 / (2 * sigma_range**2))
            weight = spatial_weight * range_weight
            out[i, j] = (weight * patch).sum() / weight.sum()
    return out

### Đối chiếu với OpenCV

`cv2.bilateralFilter` dùng cùng ý tưởng với một số phép xấp xỉ bên trong để tăng tốc, nên ta kỳ vọng kết quả gần giống nhưng không khớp chính xác đến từng điểm ảnh.

In [ ]:
step = np.zeros((40, 40), dtype=np.uint8)
step[20:, :] = 200
noisy_step = np.clip(step.astype(np.float64) + rng.normal(0, 10, step.shape), 0, 255).astype(np.uint8)

mine = bilateral_filter(noisy_step, radius=3, sigma_spatial=3, sigma_range=25)
cv_result = cv2.bilateralFilter(noisy_step, d=7, sigmaColor=25, sigmaSpace=3).astype(np.float64)

print(f'max abs difference  = {np.abs(mine - cv_result).max():.2f} gray levels')
print(f'mean abs difference = {np.abs(mine - cv_result).mean():.2f} gray levels')

### Bảo toàn biên, đặt cạnh nhau

Phép so sánh thực sự quan trọng: trên một biên bậc bị nhiễu, bộ lọc có làm trơn nhiễu trong khi giữ biên sắc nét, hay làm mờ biên cùng với nhiễu?

In [ ]:
big_step = np.zeros((150, 150), dtype=np.uint8)
big_step[75:, :] = 200
noisy_big = np.clip(big_step.astype(np.float64) + rng.normal(0, 15, big_step.shape), 0, 255).astype(np.uint8)

gaussian_smoothed = cv2.GaussianBlur(noisy_big, (0, 0), sigmaX=4)
bilateral_smoothed = cv2.bilateralFilter(noisy_big, d=9, sigmaColor=40, sigmaSpace=15)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, im, title in zip(axes[0], [noisy_big, gaussian_smoothed, bilateral_smoothed],
                          ['Noisy step edge', 'Gaussian blur', 'Bilateral filter']):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

col = 75
for ax, im, title in zip(axes[1], [noisy_big, gaussian_smoothed, bilateral_smoothed],
                          ['profile: noisy', 'profile: Gaussian', 'profile: bilateral']):
    ax.plot(im[:, col])
    ax.set_ylim(0, 255)
    ax.set_title(title, fontsize=9)
plt.tight_layout()
plt.show()

Biên dạng (profile) sau làm mờ Gauss cho thấy một dốc thoải trải qua nhiều hàng &mdash; biên đã bị làm mềm thấy rõ. Biên dạng sau lọc song phương vẫn phẳng ở mỗi bên (nhiễu đã bị loại bỏ) nhưng vẫn nhảy vọt gắt gần hàng 75 &mdash; biên được giữ lại vì các điểm ảnh ở hai bên ranh giới có cường độ quá khác nhau để có thể bị lấy trung bình cùng nhau, bất kể chúng gần nhau về không gian đến đâu.

### Bài tập

1. Áp dụng bộ lọc trung vị (thay vì Gauss hay song phương) lên `noisy_big` và thêm biên dạng của nó vào phép so sánh. Nó so với bộ lọc song phương thế nào trong việc bảo toàn biên?
2. Trong `bilateral_filter`, đặt `sigma_range` rất lớn (ví dụ 1000). Bộ lọc khi đó sẽ suy biến thành gì, và kết quả của bạn có xác nhận điều đó không?
3. Lọc song phương chậm hơn nhiều so với làm mờ Gauss vì các trọng số phải được tính lại tại mỗi điểm ảnh (chúng phụ thuộc vào cường độ cục bộ, không chỉ vị trí). Đo thời gian `bilateral_filter` tự viết so với `cv2.bilateralFilter` trên ảnh 100x100 bằng `%timeit`, và mô tả bằng lời cách cài đặt của OpenCV có thể làm khác đi như thế nào để nhanh hơn.